# Building Dimensions table for the constructors silver tables

In [0]:
%run ../00.Common/01.Environment-config

In [0]:
target_name = f"{catalog_name}.{gold_schema}.dim_constructors"
constructors_table = f"{catalog_name}.{silver_schema}.constructors"
nationality_table = f"{catalog_name}.{gold_schema}.ref_nationality_region"

###  Reading the silver tables

In [0]:
constructors_df = spark.table(constructors_table)
nationality_df = spark.table(nationality_table)

### Join the nationality dataframe with the constructors dataframe and renaming the region column name

In [0]:
dim_constructors_df = (
    constructors_df.join(
        nationality_df, constructors_df.nationality == nationality_df.nationality, "left"
        ).select(constructors_df.constructor_id, 
                 constructors_df.constructor_name,
                 constructors_df.nationality, 
                 nationality_df.region.alias("nationality_region"))
)

In [0]:
display(dim_constructors_df)

### Write the table into gold schema

In [0]:
(
    dim_constructors_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(target_name)
)

In [0]:
display(spark.table(target_name))